# NLP Assignment II: Open-Domain Question Answering (From Lexical Retrieval to RAG)
**Course:** CS60075 Natural Language Processing, Autumn 2026, IIT Kharagpur  
**Group:** 29 | **Project:** 10  
**Assignment:** CS60075 NLP (Autumn 2026) | Group 29 | Project 10  
**Dataset:** Natural Questions (Open) [`google-research-datasets/nq_open`]

---

### End-to-End Google Colab Walkthrough
This notebook demonstrates the complete three-task open-domain QA pipeline:
1. **Task 1: Classical Lexical Retrieve-and-Extract** (BM25Okapi + Syntactic Chunk Parsing)
2. **Task 2: Dense Retrieval-Augmented Generation** (MiniLM + FAISS Flat IP + Qwen2.5-1.5B-Instruct)
3. **Task 3: Hybrid Retrieval with Groundedness Verification** (Reciprocal Rank Fusion c=60 + Lexical Overlap Filtering)

## 1. Setup Environment & Dependencies
Run the following cell to install the required libraries.

In [ ]:
!pip install -q rank-bm25 sentence-transformers faiss-cpu transformers accelerate nltk pandas matplotlib

import os, sys, json, re, string
from collections import Counter, defaultdict
import numpy as np
import torch
import nltk
from tqdm.auto import tqdm

for res in ['punkt', 'averaged_perceptron_tagger', 'punkt_tab']:
    nltk.download(res, quiet=True)

print('Environment initialized successfully.')
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())

## 2. Load Dataset & Frozen 45k Wikipedia Passage Index
We load the 300 sampled validation questions (seed 42) and the frozen 45,029 Wikipedia passage index.

In [ ]:
# If running inside cloned repo
if os.path.exists('data/sampled_questions.jsonl'):
    with open('data/sampled_questions.jsonl', 'r', encoding='utf-8') as f:
        questions = [json.loads(line) for line in f]
    with open('data/nq_open_n300_seed42.jsonl', 'r', encoding='utf-8') as f:
        corpus = [json.loads(line) for line in f]
    print(f'Loaded {len(questions)} evaluation questions and {len(corpus)} passages from local repository.')
else:
    print('Please ensure data/ is present in the current working directory.')

## 3. Evaluation Metrics Implementation
SQuAD-style normalization, Exact Match (EM), Token-F1, and Recall@k.

In [ ]:
def normalize_answer(text):
    if not text:
        return ''
    text = text.lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = ''.join(ch for ch in text if ch not in set(string.punctuation))
    return ' '.join(text.split())

def compute_exact_match(prediction, ground_truths):
    norm_pred = normalize_answer(prediction)
    for gt in ground_truths:
        if norm_pred == normalize_answer(gt):
            return True
    return False

def compute_f1(prediction, ground_truths):
    pred_tokens = normalize_answer(prediction).split()
    if not pred_tokens:
        return 0.0
    best_f1 = 0.0
    for gt in ground_truths:
        gt_tokens = normalize_answer(gt).split()
        if not gt_tokens:
            continue
        common = Counter(pred_tokens) & Counter(gt_tokens)
        num_same = sum(common.values())
        if num_same == 0:
            continue
        precision = 1.0 * num_same / len(pred_tokens)
        recall = 1.0 * num_same / len(gt_tokens)
        f1 = (2 * precision * recall) / (precision + recall)
        if f1 > best_f1:
            best_f1 = f1
    return best_f1

def compute_recall_at_k(retrieved_passages, ground_truths, k=5):
    norm_gts = [normalize_answer(gt) for gt in ground_truths if normalize_answer(gt)]
    for passage in retrieved_passages[:k]:
        norm_p = normalize_answer(passage)
        for gt in norm_gts:
            if gt in norm_p:
                return True
    return False

## 4. Task 1: Classical Lexical Baseline (BM25 + Extractive QA)
- BM25Okapi ($k_1=1.5, b=0.75$)
- Highest unigram lexical overlap sentence selection
- Longest noun phrase or number span extraction via NLTK chunk parser

In [ ]:
from rank_bm25 import BM25Okapi

NP_GRAMMAR = r'NP: {<DT|PRP\$>?<JJ.*|VBG|VBN>*<NN.*>+}'
NP_PARSER = nltk.RegexpParser(NP_GRAMMAR)
NUM_REGEX = re.compile(r'\b(?:\d{1,4}(?:,\d{3})*(?:\.\d+)?|\d+(?:st|nd|rd|th)?)\b')

def extract_longest_candidate(sentence):
    if not sentence or not sentence.strip():
        return ''
    candidates = []
    try:
        words = nltk.word_tokenize(sentence)
        pos_tags = nltk.pos_tag(words)
        tree = NP_PARSER.parse(pos_tags)
        for subtree in tree.subtrees():
            if subtree.label() == 'NP':
                candidates.append(' '.join(leaf[0] for leaf in subtree.leaves()))
    except Exception:
        pass
    candidates.extend(NUM_REGEX.findall(sentence))
    return max(candidates, key=len).strip() if candidates else sentence.strip()

def select_best_sentence(question, passage_text):
    try:
        sentences = nltk.sent_tokenize(passage_text)
    except Exception:
        sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', passage_text) if s.strip()]
    if not sentences:
        return passage_text
    q_words = set(re.findall(r'\b\w+\b', question.lower()))
    best_sent = sentences[0]
    max_overlap = -1
    for sent in sentences:
        sent_words = set(re.findall(r'\b\w+\b', sent.lower()))
        overlap = len(q_words & sent_words)
        if overlap > max_overlap:
            max_overlap = overlap
            best_sent = sent
    return best_sent

## 5. Task 2: Dense Retrieval-Augmented Generation (RAG Reproduction)
- Bi-encoder: `sentence-transformers/all-MiniLM-L6-v2` ($d=384$)
- FAISS Flat Inner-Product Index (exact cosine similarity)
- Generator: `Qwen/Qwen2.5-1.5B-Instruct` (greedy, temp=0, max_new_tokens=64)

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using compute device: {device.upper()}')

# Precomputed embeddings can be loaded directly from data/dense_embeddings_45k.npy
if os.path.exists('data/dense_embeddings_45k.npy'):
    corpus_embeddings = np.load('data/dense_embeddings_45k.npy')
    print(f'Loaded cached dense embeddings matrix with shape: {corpus_embeddings.shape}')
    faiss_index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
    faiss_index.add(corpus_embeddings.astype(np.float32))
    print('FAISS index ready.')

## 6. Task 3: Hybrid Retrieval (RRF) & Groundedness Verification
- Reciprocal Rank Fusion ($c=60$)
- Lexical overlap groundedness verification ($S_{\text{overlap}} \ge 0.5$)

In [ ]:
def compute_reciprocal_rank_fusion(bm25_ranked_ids, dense_ranked_ids, constant=60):
    rrf_scores = defaultdict(float)
    for rank_idx, doc_id in enumerate(bm25_ranked_ids):
        rrf_scores[doc_id] += 1.0 / (constant + rank_idx + 1)
    for rank_idx, doc_id in enumerate(dense_ranked_ids):
        rrf_scores[doc_id] += 1.0 / (constant + rank_idx + 1)
    return sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)

STOPWORDS = {'a', 'an', 'the', 'in', 'on', 'at', 'of', 'for', 'to', 'and', 'or', 'is', 'was', 'are', 'were', 'it', 'its', 'that', 'this', 'by', 'from', 'with', 'as', 'be', 'has', 'had', 'have'}

def compute_lexical_overlap(answer, context):
    norm_tokens = normalize_answer(answer).split()
    content_tokens = [t for t in norm_tokens if t not in STOPWORDS]
    if not content_tokens:
        return 1.0
    ctx_tokens = set(normalize_answer(context).split())
    matched = sum(1 for t in content_tokens if t in ctx_tokens)
    return float(matched) / len(content_tokens)

## 7. Comparative Benchmark Summary & Results
Load the consolidated benchmark metrics across all three tasks.

In [ ]:
import pandas as pd

if os.path.exists('data/results/consolidated_results.csv'):
    df = pd.read_csv('data/results/consolidated_results.csv')
    print(df.to_string(index=False))
else:
    print('Run python run_all.py --eval_only to generate the summary table.')